# Project Summary: Group Buy Now, Pay Later — Approach, Issues, and Assumptions

This notebook summarizes the end-to-end approach, key decisions, issues encountered, and limitations/assumptions made throughout the project.

## Objectives

- **Business goal**: Rank merchants for onboarding and monitoring by combining revenue potential and expected fraud risk.
- **Technical goal**: Build a reproducible Spark pipeline to clean/curate transactions, estimate per-transaction fraud probabilities, aggregate to merchant KPIs, and produce composite rankings and segment insights.

## Data Sources

- `data/tables/transaction_data/` three snapshots of transactions in Parquet.
- `data/tables/merchant_data/` merchant metadata (`tbl_merchants.parquet`, `consumer_user_details.parquet`, fraud probabilities CSVs).
- `data/income/` SA2-level income and a locality-to-SA2 index, used for exploratory socioeconomic context.
- Curated outputs in `data/curated/` and final rankings in `data/final_ranks/`.

Relevant notebooks:
- `notebooks/01_etl.ipynb`: curation, outlier handling, segmentation, joins.
- `notebooks/02_fraud_model.ipynb`: fraud probability tiers, modeling, calibration, EB aggregation, plots.
- `notebooks/03_final_rankings.ipynb`: composite ranking combining takings, loss potential, and growth.
- `notebooks/06_income_outlier.ipynb`: SA2 income analysis and binning.
- `notebooks/04_plots.ipynb`: exploratory plots for distributions and segments.
- `notebooks/05_time_series.ipynb`: time-series exploration of transactions.



## End-to-End Approach

1. **Curation & Cleaning (Spark)**
   - Loaded three transaction snapshots; unified with `unionByName`.
   - Parsed `tbl_merchants.tags` into `biz_tags`, `rev_band`, and numeric `take_rate`.
   - Removed NULLs and filtered outliers per `biz_tags` using IQR with a dataset-size scaling factor.
   - Segmented merchants into 5 macro-segments (e.g., Technology & Professional Services, Home, Garden & Living).
   - Wrote curated datasets: `merchant_transactions/` and `agg_by_userbiz/`.

2. **Fraud Probability Estimation (Tiered)**
   - Tier A: Direct match of `consumer_fraud_probability` by `(user_id, order_date)` → `p_direct`.
   - Tier B: For missing `p_direct`, computed user-level exponential time-decay interpolation → `p_user_decay`.
   - Tier C: For remaining rows, trained LR/GBT regressors on engineered features; clipped and used `p_model_calibrated`.
   - Coalesced final per-transaction probability `p_hat = coalesce(p_direct, p_user_decay, p_model_calibrated)` with global-mean fallback and clipping to [0,1].

3. **Diagnostics & Calibration**
   - Validated regression with MAE/Brier and plotted calibration curve, lift-by-decile, and error distribution.
   - Produced segment EFLR bar chart.

4. **Merchant Aggregation & Ranking**
   - Aggregated per-merchant: counts, sums, mean risk, expected fraud loss (EFL), loss rate (EFLR).
   - Applied Empirical Bayes shrinkage to stabilize mean risk (`eb_p`) with κ prior.
   - Computed composite ranking combining: Takings Rank, Loss Potential Ranking, Growth Potential Ranking.
   - Exported: `final_merchant_rankings.csv` and per-segment rankings.


## Key Issues Encountered

- **Parsing merchant `tags`**: The raw `tags` field contained nested brackets and inconsistent spacing. Resolved with regex-based cleaning and split rules to reliably extract `biz_tags`, `rev_band`, and `take_rate`.
- **Outliers in `dollar_value`**: Extremely large amounts skewed aggregates. Applied IQR-based filtering per `biz_tags` with a scale factor `√(ln n) - 0.5`, and lower-bound clamped at 0 when needed.
- **Join sparsity for `p_direct`**: Many transactions lacked a direct probability match. Introduced Tier B (user-level decay) and Tier C (model) to cover the gap.
- **Model calibration and clipping**: Regression outputs can exceed [0,1]. We clipped predictions and validated with calibration curves and Brier score.
- **Window operations without partition keys**: Ranking steps triggered single-partition warnings. Acceptable for local runs but would require partition strategies for scale.
- **Income data ambiguity**: SA2↔locality mappings are many-to-many; used averages and exploratory binning for context only (not fed into the core model).



## Assumptions

- **Fraud probabilities**: Input `fraud_probability` is well-defined per `(user_id, order_date)` and roughly calibrated; if given in percent, we normalize to [0,1].
- **Time decay**: Exponential decay with γ=0.03 captures recency; alternative kernels were not evaluated due to time.
- **Coalescing rule**: Priority is Tier A → Tier B → Tier C; if all missing, fallback to global mean.
- **EB shrinkage**: Global prior `μ` and scalar `κ` stabilize merchants with low volume; κ chosen heuristically.
- **Take rate**: Parsed from merchant metadata; assumed correct and stable across the observation window.
- **Segments**: Macro-segments are a simplified mapping of `biz_tags`, sufficient for reporting.



## Limitations

- **Model form**: Regression (LR/GBT) on engineered features; does not capture sequential behavior beyond windowed aggregates.
- **Potential leakage**: Care taken to use only same-day joins for probabilities; however, productionization would require strict temporal validation.
- **Single-machine execution**: Local Spark settings and single-partition warnings indicate the need for distributed tuning for larger scale.
- **Income context**: Income features were not integrated into the core fraud model due to mapping ambiguity and scope constraints.
- **Composite ranking**: Linear average of three ranks is simple; weights and methodology could be tuned against business objectives.



## Outputs

- Curated tables in `data/curated/`:
  - `merchant_transactions/`: cleaned and segmented transactions (post outlier handling).
  - `agg_by_userbiz/`: per user–merchant aggregates.
  - `best_100_merchants.csv`: safest merchants by EB-adjusted probability.
  - `merchant_fraud_rankings.csv`: full EB-aggregated merchant metrics.
- Final composite rankings in `data/final_ranks/`:
  - `final_merchant_rankings.csv` and per-segment CSVs.
- Plots in `plots/`:
  - `calibration_curve.png`, `lift_decile.png`, `error_distribution.png`, `segment_eflr_bar.png`.



## Reproducibility Notes

- Execution environment: PySpark locally with tuned memory/shuffle settings.
- Determinism: Random seeds used for splits where applicable; window aggregations and joins are deterministic given inputs.
- Idempotency: Curated outputs overwrite in `data/curated/` and `data/final_ranks/` to maintain a clean state.
- Next steps for production: orchestrate as modular Spark jobs, introduce temporal CV and model monitoring, and partition strategies for window ops.

